# Workshop 1 - Gold KPI Package (exercise)

![Workshop success criteria](../../assets/images/workshop_success_criteria.png)

## Scenario

RetailHub needs a new daily channel reporting slice for Power BI. You extend the Gold layer, define additional KPI, inspect data-quality issues, reconcile aggregates and document the reporting-source decision.

## Objectives

After completing this workshop you will be able to:

- build a new Gold reporting table with a documented grain,
- define KPI that are not already covered by the Day 1 demo,
- find data-quality issues using concrete example rows,
- reconcile two aggregates that should agree,
- fill a decision log entry for a table-vs-view-vs-aggregate choice.

## Prerequisites

Before starting, complete:

1. `setup/00_setup.ipynb`
2. `notebooks/demo/day1_02_lakehouse_delta_gold.ipynb`
3. `notebooks/demo/day1_03_gold_modeling_for_powerbi.ipynb`

## Setup

Initialize the environment, resolve the participant catalog and expose the shared variables used by this workshop.

In [ ]:
%run ../../setup/00_setup

### Configuration

Display the active RetailHub catalog context and validate prerequisite objects before starting the tasks.

In [ ]:
display(spark.createDataFrame([
    ("CATALOG", CATALOG),
    ("BRONZE", BRONZE),
    ("SILVER", SILVER),
    ("GOLD", GOLD),
], ["Variable", "Value"]))

### Runtime Pre-check

Fail fast if an upstream demo notebook has not created the required Gold objects.

In [ ]:
required_tables = [
    f"{GOLD}.fact_sales_dashboard",
    f"{GOLD}.fact_sales_dashboard_monthly",
]

missing = [t for t in required_tables if not spark.catalog.tableExists(t)]
if missing:
    print("[BLOCKED] Missing required tables:")
    for t in missing:
        print("  -", t)
    print()
    print("Run this notebook first: notebooks/demo/day1_02_lakehouse_delta_gold.ipynb (for fact_sales_dashboard) and notebooks/demo/day1_03_gold_modeling_for_powerbi.ipynb (for fact_sales_dashboard_monthly)")
    raise Exception("Pre-check failed: missing prerequisite tables. Run \"notebooks/demo/day1_02_lakehouse_delta_gold.ipynb (for fact_sales_dashboard) and notebooks/demo/day1_03_gold_modeling_for_powerbi.ipynb (for fact_sales_dashboard_monthly)\" first.")

print("[OK] Pre-check passed, all required tables exist:")
for t in required_tables:
    print("  -", t)

## Success Criteria

You are done when:

1. `gold.fact_sales_dashboard_channel_daily` exists with the documented grain.
2. Average Order Value, Margin Rate % and Completed Share by Region are
   calculated from Gold - these are NOT the same KPI the Day 1 demo already showed
   you (Revenue, Gross Margin, Return Rate, Orders) - you are extending the
   dictionary, not repeating it.
3. At least three data-quality issues are named with example rows, using the
   bad-data-lab patterns from the data generator.
4. Two aggregates that should agree are reconciled, and any mismatch is
   explained.
5. The decision log (`docs/templates/decision-log.md`) has a real, filled-in
   row for this workshop's reporting-source decision.
6. The self-check cell passes.

## Prerequisite Chain

This workshop is the last link in a chain. If the pre-check above failed,
walk back through the chain in order:

1. `setup/00_pre_config.ipynb`
2. `data/generate_training_dataset.ipynb`
3. `notebooks/demo/day1_02_lakehouse_delta_gold.ipynb` - build the first Gold detail contract.
4. `notebooks/demo/day1_03_gold_modeling_for_powerbi.ipynb` - uruchom Gold modeling demo najpierw,
   jesli `gold.fact_sales_dashboard` lub `gold.fact_sales_dashboard_monthly`
   nie istnieja. Ten warsztat startuje dokladnie tam, gdzie demo Gold semantic layer sie konczy.

## The 5 Tasks

1. **Zadanie 1 - zbuduj reporting table.** Build a new Gold object,
   `gold.fact_sales_dashboard_channel_daily`, one level more granular in time
   (daily, not monthly) and narrower in dimensions (channel only) than
   anything the Day 1 demo built - a realistic "BI asked for a daily channel view"
   request.
2. **Zadanie 2 - zdefiniuj KPI.** Define four KPI that the Day 1 demo dictionary
   does NOT already contain: Average Order Value, Margin Rate %, Completed
   Share by Region and Revenue per Channel-Day.
3. **Zadanie 3 - znajdz data quality issues.** Find at least three
   data-quality issues directly in `gold.fact_sales_dashboard`, with example
   rows, using the bad-data-lab patterns.
4. **Zadanie 4 - zrob reconciliation.** Compare your new daily-channel
   aggregate against `gold.fact_sales_dashboard_monthly` for the same scope
   and prove they agree (or explain why they do not).
5. **Zadanie 5 - wypelnij decision log.** Fill a real row in the decision log
   for the table-vs-view-vs-aggregate choice you made in Task 1.

## Zadanie 1 - zbuduj reporting table

Build `gold.fact_sales_dashboard_channel_daily`: one row per `order_date` x
`channel`, with completed orders, revenue and gross margin, sourced from
`gold.fact_sales_dashboard`. Decide table vs view (you will justify the
choice in Task 5).

**Oczekiwany wynik (rubric):**
- table exists with columns `order_date, channel, completed_orders, revenue, gross_margin`,
- `COUNT(*) == COUNT(DISTINCT (order_date, channel))` - grain is unique,
- minimum success: the table is queryable and non-empty.

In [ ]:
# TODO: create gold.fact_sales_dashboard_channel_daily
# grain: one row per (order_date, channel)
# columns: completed_orders, revenue, gross_margin (completed orders only)
spark.sql(f"""
CREATE OR REPLACE TABLE {GOLD}.fact_sales_dashboard_channel_daily
COMMENT 'TODO: add a comment describing the grain and refresh choice'
AS
SELECT
  order_date,
  channel
  -- TODO: add completed_orders, revenue, gross_margin aggregates
FROM {GOLD}.fact_sales_dashboard
GROUP BY order_date, channel
""")

spark.sql(f"""
SELECT COUNT(*) AS rows, COUNT(DISTINCT (order_date, channel)) AS distinct_grain_keys
FROM {GOLD}.fact_sales_dashboard_channel_daily
""").show()

## Zadanie 2 - zdefiniuj KPI

Define FOUR KPI that the Day 1 demo did NOT already give you:

1. Average Order Value (revenue / distinct completed orders - watch the
   distinct-count trap from the Day 1 demo),
2. Margin Rate % (gross margin as a percentage of revenue, computed AFTER
   aggregation, not per-row),
3. Completed Share by Region (completed orders / all orders, per
   `customer_region`),
4. Revenue per Channel-Day (from your Task 1 table).

**Oczekiwany wynik (rubric):**
- AOV and Margin Rate % are single numbers, both NULL-safe (`NULLIF` on the
  denominator),
- Completed Share by Region returns one row per region, share between 0 and 100,
- minimum success: all four KPI use `COUNT(DISTINCT order_id)` where the
  business definition needs a per-order count, never raw `COUNT(*)`.

In [ ]:
# TODO: Average Order Value and Margin Rate %
spark.sql(f"""
SELECT
  -- TODO: avg_order_value = completed revenue / distinct completed orders
  -- TODO: margin_rate_pct = 100 * completed margin / completed revenue
  COUNT(*) AS rows
FROM {GOLD}.fact_sales_dashboard
""").show()

In [ ]:
# TODO: Completed Share by Region
spark.sql(f"""
SELECT
  customer_region
  -- TODO: add all_orders, completed_orders, completed_share_pct
FROM {GOLD}.fact_sales_dashboard
GROUP BY customer_region
""").show()

In [ ]:
# TODO: Revenue per Channel-Day, using your Task 1 table
spark.sql(f"""
SELECT order_date, channel, revenue
  -- TODO: add revenue_per_completed_order
FROM {GOLD}.fact_sales_dashboard_channel_daily
ORDER BY order_date DESC, channel
LIMIT 10
""").show()

## Zadanie 3 - znajdz data quality issues

Find at least THREE data-quality issues directly in
`gold.fact_sales_dashboard` (not Silver). Hint: a left join does not delete
orphan references, it turns them into NULLs - check `customer_region` and
`category` for NULLs in addition to the missing-price/invalid-status/
future-date checks you already know from the Day 1 demo.

**Oczekiwany wynik (rubric):**
- at least 3 distinct `issue_type` rows with `issue_count > 0` for at least
  some of them (a generated dataset should reproduce the seeded bad-data-lab
  problems),
- a second query returns concrete example rows (not just counts) for at
  least one issue type,
- minimum success: every issue type you list is checked against
  `gold.fact_sales_dashboard`, with an explanation of what it means at the
  Gold layer (not just "this is a Silver bug").

In [ ]:
# TODO: count at least 3 data-quality issue types against gold.fact_sales_dashboard.
# Include at least one check that did NOT appear in the demo DQ score
# (e.g. null customer_region or null category after the join).
spark.sql(f"""
SELECT status, COUNT(*) AS rows
FROM {GOLD}.fact_sales_dashboard
GROUP BY status
ORDER BY rows DESC
""").show()

In [ ]:
# TODO: pull concrete example rows for at least one issue type found above.
spark.sql(f"""
SELECT line_id, order_id, order_date, status, customer_region, category, unit_price
FROM {GOLD}.fact_sales_dashboard
LIMIT 20
""").show(truncate=False)

## Zadanie 4 - zrob reconciliation

Compare your `gold.fact_sales_dashboard_channel_daily` (Task 1), rolled up to
month + channel, against `gold.fact_sales_dashboard_monthly` (Day 1 demo),
also rolled up to month + channel. Revenue must match for every
(year_month, channel) pair.

**Oczekiwany wynik (rubric):**
- a result set with `daily_rollup_revenue`, `monthly_table_revenue` and
  `diff` columns,
- `diff` is 0 or within rounding tolerance (< 0.01) for every row,
- minimum success: if `diff` is NOT close to zero, you can name the reason
  (different filter, different grain, double-counted join) rather than
  ignoring the mismatch.

In [ ]:
# TODO: roll up gold.fact_sales_dashboard_channel_daily to (year_month, channel)
# and compare revenue against gold.fact_sales_dashboard_monthly rolled up the same way.
spark.sql(f"""
SELECT 'TODO' AS year_month, 'TODO' AS channel, 0.0 AS daily_rollup_revenue, 0.0 AS monthly_table_revenue, 0.0 AS diff
""").show()

## Zadanie 5 - wypelnij decision log

Fill a real row of `docs/templates/decision-log.md` for the table-vs-view
choice you made in Task 1. Write it as if a teammate will read it in six
months with no other context.

**Oczekiwany wynik (rubric):**
- every column filled (Date, Decision, Options considered, Chosen option,
  Reason, Consequence) - no blanks,
- "Reason" references a concrete tradeoff (cost, freshness, refresh
  orchestration), not just "it seemed best",
- minimum success: "Consequence" names at least one thing that becomes true
  *because* of the choice (e.g. "needs a scheduled job").

| Date | Decision | Options considered | Chosen option | Reason | Consequence |
|---|---|---|---|---|---|
| TODO | TODO | TODO | TODO | TODO | TODO |

## Bonus (dla szybszych grup)

- add `segment` as a third grouping column to your Task 1 table and re-check
  the Task 4 reconciliation still holds,
- write a `COMMENT ON COLUMN` for `revenue` on your new table,
- prepare a one-minute explanation of your Task 3 findings for a business
  stakeholder who has never heard the term "orphan reference".

## Summary

In this workshop you extend the Day 1 Gold layer with a new reporting slice, define additional KPI, validate data-quality issues, reconcile aggregates and document the reporting-source decision.

The expected output is not only SQL that runs. The expected output is a BI-ready Gold artifact that another Power BI developer can trust and reuse.